In [ ]:
import pandas as pd
from db import engine, admin_engine
from style import setup_style
from stop_words import STOP_WORDS
from news_helpers import (
    compute_embeddings, load_embeddings, compute_umap, plot_by_source,
    run_bertopic, save_topics, show_topics, plot_umap_by_topic, plot_topic_timeline,
)
setup_style()

MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
BATCH_SIZE = 256
RUN_ID     = "run_01"
NEWS_LANG  = "all"

total    = pd.read_sql("SELECT count(*) AS n FROM news_events", engine).iloc[0]["n"]
with_emb = pd.read_sql("SELECT count(*) AS n FROM news_events WHERE embedding IS NOT NULL", engine).iloc[0]["n"]
print(f"Artigos: {total:,}  ·  com embedding: {with_emb:,}")

In [ ]:
compute_embeddings(engine, admin_engine, MODEL_NAME, BATCH_SIZE)

In [ ]:
df_emb, X = load_embeddings(engine)

In [ ]:
df_emb = compute_umap(df_emb, X)
plot_by_source(df_emb)

In [ ]:
df_emb, topic_model = run_bertopic(df_emb, X, STOP_WORDS)
save_topics(df_emb, topic_model, RUN_ID, MODEL_NAME, NEWS_LANG, str(len(STOP_WORDS)) + "_words", admin_engine)

In [ ]:
show_topics(engine)
plot_umap_by_topic(df_emb)
plot_topic_timeline(df_emb)